In [3]:
# ============================================================
# THIRD STRATEGY
# INTEGRATED GRADIENTS (IG) GENE SELECTION
# + GENEFORMER TOKENIZATION
#
# Pipeline:
#
# Training data
#       ↓
# Logistic Regression
#       ↓
# Integrated Gradients
#       ↓
# ALL training samples
#       ↓
# Mean |IG| gene importance
#       ↓
# Top 510 / 1022 / 2046 / 4094 genes
#       ↓
# Geneformer expression ranking
#       ↓
# Geneformer token sequences
#       ↓
# Hugging Face Dataset
#
# IMPORTANT:
# IG uses ALL training samples.
# Validation and test are NEVER used for gene selection.
# ============================================================


In [15]:
# ============================================================
# 0. IMPORTS
# ============================================================

import geneformer
import inspect

print("=" * 70)
print("GENEFORMER VERSION / TOKENIZER CHECK")
print("=" * 70)

print("Geneformer location:")
print(geneformer.__file__)

print("\nGeneformer version:")
print(
    getattr(
        geneformer,
        "__version__",
        "version not exposed"
    )
)

from geneformer import TranscriptomeTokenizer

print("\nTranscriptomeTokenizer:")
print(TranscriptomeTokenizer)

print("\nConstructor:")
print(
    inspect.signature(
        TranscriptomeTokenizer
    )
)


# ============================================================
# GENERAL IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import pickle
import json
import joblib

from pathlib import Path
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict

from sklearn.linear_model import LogisticRegression


GENEFORMER VERSION / TOKENIZER CHECK
Geneformer location:
F:\GeneformerProject\Geneformer\geneformer\__init__.py

Geneformer version:
version not exposed

TranscriptomeTokenizer:
<class 'geneformer.tokenizer.TranscriptomeTokenizer'>

Constructor:
(custom_attr_name_dict=None, nproc=1, chunk_size=512, model_input_size=4096, special_token=True, collapse_gene_ids=True, use_h5ad_index=False, keep_counts=False, model_version='V2', gene_median_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/gene_median_dictionary_gc104M.pkl'), token_dictionary_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/token_dictionary_gc104M.pkl'), gene_mapping_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/ensembl_mapping_dict_gc104M.pkl'))


In [16]:
# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path(
    r"F:\GeneformerProject\data\geneformer_ready\preprocessed_5_cancers"
)


# IMPORTANT:
# This is THIRD strategy, so use a separate directory.

OUTPUT_DIR = Path(
    r"F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


In [17]:
# ============================================================
# 2. PARAMETERS
# ============================================================

# These are Geneformer sequence lengths.

TOP_K = [
    512,
    1024,
    2048,
    4096
]


# Number of genes in each sequence:
#
# 512  = CLS + 510 genes + EOS
# 1024 = CLS + 1022 genes + EOS
# 2048 = CLS + 2046 genes + EOS
# 4096 = CLS + 4094 genes + EOS

N_GENES = [
    510,
    1022,
    2046,
    4094
]


MAX_LENGTH = TOP_K


In [18]:
# ------------------------------------------------------------
# Geneformer vocabulary
# ------------------------------------------------------------

TOKEN_FILE = Path(
    r"F:\GeneformerProject\Geneformer\geneformer"
    r"\token_dictionary_gc104M.pkl"
)

In [19]:
# ------------------------------------------------------------
# Input datasets
# ------------------------------------------------------------

TRAIN_FILE = (
    BASE_DIR /
    "train_geneformer.parquet"
)

VAL_FILE = (
    BASE_DIR /
    "validation_geneformer.parquet"
)

TEST_FILE = (
    BASE_DIR /
    "test_geneformer.parquet"
)




LABEL_COLUMNS = [
    "Cancer_Type",
    "tissue_type"
]


RANDOM_STATE = 42


# ============================================================
# 3. LOAD GENEFORMER VOCABULARY
# ============================================================

print("\n" + "=" * 70)
print("LOADING GENEFORMER VOCABULARY")
print("=" * 70)


with open(
    TOKEN_FILE,
    "rb"
) as f:

    token_dictionary = pickle.load(f)


CLS_TOKEN = token_dictionary["<cls>"]

EOS_TOKEN = token_dictionary["<eos>"]


print(
    "Geneformer vocabulary size:",
    len(token_dictionary)
)

print(
    "CLS token:",
    CLS_TOKEN
)

print(
    "EOS token:",
    EOS_TOKEN
)

print(
    "Maximum sequence lengths:",
    MAX_LENGTH
)

print(
    "Genes per sequence:",
    N_GENES
)


# ============================================================
# 4. LOAD DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING DATASETS")
print("=" * 70)


train_df = pd.read_parquet(
    TRAIN_FILE
)

val_df = pd.read_parquet(
    VAL_FILE
)

test_df = pd.read_parquet(
    TEST_FILE
)


print("\nDataset sizes:")

print(
    "Train:",
    train_df.shape
)

print(
    "Validation:",
    val_df.shape
)

print(
    "Test:",
    test_df.shape
)


# ============================================================
# 5. IDENTIFY GENE COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("IDENTIFYING GENES")
print("=" * 70)


gene_columns = [
    c
    for c in train_df.columns
    if c not in LABEL_COLUMNS
]


print(
    "Number of genes:",
    len(gene_columns)
)


assert len(gene_columns) == 15100, (
    f"Expected 15100 genes, "
    f"found {len(gene_columns)}"
)


# ============================================================
# 6. CHECK TRAIN / VALIDATION / TEST GENE COLUMNS
# ============================================================

val_gene_columns = [
    c
    for c in val_df.columns
    if c not in LABEL_COLUMNS
]


test_gene_columns = [
    c
    for c in test_df.columns
    if c not in LABEL_COLUMNS
]


assert gene_columns == val_gene_columns, (
    "Validation gene columns do not match training."
)


assert gene_columns == test_gene_columns, (
    "Test gene columns do not match training."
)


print(
    "✓ Train/validation/test gene columns match"
)


# ============================================================
# 7. CHECK GENEFORMER VOCABULARY
# ============================================================

print("\n" + "=" * 70)
print("CHECKING GENEFORMER VOCABULARY")
print("=" * 70)


missing_genes = [
    gene
    for gene in gene_columns
    if gene not in token_dictionary
]


print(
    "Genes missing from Geneformer vocabulary:",
    len(missing_genes)
)


assert len(missing_genes) == 0, (
    f"{len(missing_genes)} genes are missing "
    "from Geneformer vocabulary."
)


print(
    "✓ All genes are present in Geneformer vocabulary"
)


# ============================================================
# 8. CREATE CANCER LABEL MAPPING
# ============================================================

print("\n" + "=" * 70)
print("CREATING CANCER LABEL MAPPING")
print("=" * 70)


# IMPORTANT:
# Mapping is created ONLY from training data.

cancer_classes = sorted(
    train_df[
        "Cancer_Type"
    ]
    .dropna()
    .unique()
    .tolist()
)


label_to_id = {
    label: idx
    for idx, label
    in enumerate(cancer_classes)
}


id_to_label = {
    idx: label
    for label, idx
    in label_to_id.items()
}


print(
    "Number of cancer classes:",
    len(cancer_classes)
)


print(
    "\nCancer class mapping:"
)


for idx, label in id_to_label.items():

    print(
        f"{idx:3d} -> {label}"
    )


# ============================================================
# 9. CHECK VALIDATION / TEST LABELS
# ============================================================

unknown_val_labels = sorted(
    set(
        val_df["Cancer_Type"]
    )
    -
    set(label_to_id)
)


unknown_test_labels = sorted(
    set(
        test_df["Cancer_Type"]
    )
    -
    set(label_to_id)
)


assert len(unknown_val_labels) == 0, (
    "Validation contains cancer classes "
    "not present in training."
)


assert len(unknown_test_labels) == 0, (
    "Test contains cancer classes "
    "not present in training."
)


print(
    "✓ Validation labels are valid"
)

print(
    "✓ Test labels are valid"
)


# ============================================================
# 10. PREPARE TRAINING DATA FOR LOGISTIC REGRESSION
# ============================================================

print("\n" + "=" * 70)
print("STEP 1 — PREPARE TRAINING DATA")
print("=" * 70)


X_train = train_df[
    gene_columns
].to_numpy(
    dtype=np.float32
)


y_train = train_df[
    "Cancer_Type"
].map(
    label_to_id
).to_numpy(
    dtype=np.int32
)


print(
    "X_train shape:",
    X_train.shape
)

print(
    "y_train shape:",
    y_train.shape
)


# ------------------------------------------------------------
# Check data
# ------------------------------------------------------------

assert np.all(
    np.isfinite(X_train)
), (
    "Training data contains "
    "NaN or infinite values."
)


assert np.all(
    np.isfinite(y_train)
)


print(
    "✓ Training data contains finite values"
)



LOADING GENEFORMER VOCABULARY
Geneformer vocabulary size: 20275
CLS token: 2
EOS token: 3
Maximum sequence lengths: [512, 1024, 2048, 4096]
Genes per sequence: [510, 1022, 2046, 4094]

LOADING DATASETS

Dataset sizes:
Train: (2411, 15102)
Validation: (517, 15102)
Test: (517, 15102)

IDENTIFYING GENES
Number of genes: 15100
✓ Train/validation/test gene columns match

CHECKING GENEFORMER VOCABULARY
Genes missing from Geneformer vocabulary: 0
✓ All genes are present in Geneformer vocabulary

CREATING CANCER LABEL MAPPING
Number of cancer classes: 5

Cancer class mapping:
  0 -> BRCA
  1 -> COAD
  2 -> KIRC
  3 -> LUAD
  4 -> PRAD
✓ Validation labels are valid
✓ Test labels are valid

STEP 1 — PREPARE TRAINING DATA
X_train shape: (2411, 15100)
y_train shape: (2411,)
✓ Training data contains finite values


In [20]:
# ============================================================
# 11. TRAIN LOGISTIC REGRESSION
# ============================================================

print("\n" + "=" * 70)
print("STEP 2 — TRAIN LOGISTIC REGRESSION")
print("=" * 70)


ig_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)


ig_model.fit(
    X_train,
    y_train
)


print(
    "✓ Logistic Regression trained"
)


print(
    "Number of classes:",
    len(
        ig_model.classes_
    )
)


print(
    "Coefficient shape:",
    ig_model.coef_.shape
)


print(
    "Intercept shape:",
    ig_model.intercept_.shape
)


# ============================================================
# 12. VERIFY MODEL CLASSES
# ============================================================

assert (
    ig_model.coef_.shape[0]
    ==
    len(cancer_classes)
)


assert (
    ig_model.coef_.shape[1]
    ==
    len(gene_columns)
)


print(
    "✓ Logistic Regression dimensions verified"
)


# ============================================================
# 13. SAVE LOGISTIC REGRESSION MODEL
# ============================================================

IG_MODEL_FILE = (
    OUTPUT_DIR /
    "ig_logistic_regression.joblib"
)


joblib.dump(
    ig_model,
    IG_MODEL_FILE
)


print(
    "Saved Logistic Regression:"
)

print(
    IG_MODEL_FILE
)



STEP 2 — TRAIN LOGISTIC REGRESSION
✓ Logistic Regression trained
Number of classes: 5
Coefficient shape: (5, 15100)
Intercept shape: (5,)
✓ Logistic Regression dimensions verified
Saved Logistic Regression:
F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\ig_logistic_regression.joblib


In [21]:
# ============================================================
# 14. DEFINE IG BASELINE
# ============================================================

print("\n" + "=" * 70)
print("STEP 3 — DEFINE INTEGRATED GRADIENTS BASELINE")
print("=" * 70)


# ------------------------------------------------------------
# IMPORTANT:
#
# Baseline is calculated ONLY from TRAINING DATA.
#
# Mean expression of every gene across all 7,742
# training samples.
# ------------------------------------------------------------

ig_baseline = X_train.mean(
    axis=0
).astype(
    np.float32
)


print(
    "IG baseline shape:",
    ig_baseline.shape
)


print(
    "Baseline mean:",
    float(
        ig_baseline.mean()
    )
)


print(
    "Baseline minimum:",
    float(
        ig_baseline.min()
    )
)


print(
    "Baseline maximum:",
    float(
        ig_baseline.max()
    )
)


assert (
    ig_baseline.shape[0]
    ==
    len(gene_columns)
)


assert np.all(
    np.isfinite(ig_baseline)
)


print(
    "✓ IG baseline verified"
)


# ============================================================
# 15. SAVE IG BASELINE
# ============================================================

IG_BASELINE_FILE = (
    OUTPUT_DIR /
    "ig_baseline_training_mean.npy"
)


np.save(
    IG_BASELINE_FILE,
    ig_baseline
)


print(
    "Saved IG baseline:"
)

print(
    IG_BASELINE_FILE
)


# ============================================================
# 16. CALCULATE INTEGRATED GRADIENTS
#     USING ALL TRAINING SAMPLES
# ============================================================

print("\n" + "=" * 70)
print("STEP 4 — CALCULATE INTEGRATED GRADIENTS")
print("=" * 70)


print(
    "Using ALL training samples:"
)

print(
    "Number of samples:",
    X_train.shape[0]
)

print(
    "Number of genes:",
    X_train.shape[1]
)

print(
    "Number of classes:",
    len(cancer_classes)
)


# ============================================================
# IMPORTANT MATHEMATICAL SIMPLIFICATION
# ============================================================
#
# Your Logistic Regression is a linear model.
#
# For class c:
#
#       F_c(x) = w_c^T x + b_c
#
# The gradient is:
#
#       dF_c/dx = w_c
#
# Therefore Integrated Gradients becomes:
#
#       IG_j =
#           (x_j - baseline_j) * w_c,j
#
# We therefore DO NOT need:
#
#       50 integration steps
#       PyTorch
#       Captum
#       sample-by-sample backward passes
#
# This makes it possible to use ALL 7,742
# training samples efficiently.
#
# ============================================================


# ------------------------------------------------------------
# Get coefficient vector corresponding to the TRUE class
# of every training sample.
#
# Shape:
#
#       (7742, 15100)
# ------------------------------------------------------------

class_specific_weights = (
    ig_model.coef_[y_train]
)


print(
    "\nClass-specific coefficient matrix:"
)

print(
    class_specific_weights.shape
)


assert (
    class_specific_weights.shape
    ==
    X_train.shape
)


# ------------------------------------------------------------
# Calculate (x - baseline)
#
# Shape:
#
#       (7742, 15100)
# ------------------------------------------------------------

X_centered = (
    X_train
    -
    ig_baseline
)


print(
    "\nCentered training matrix:"
)

print(
    X_centered.shape
)


# ------------------------------------------------------------
# Integrated Gradients
#
# Shape:
#
#       (7742, 15100)
# ------------------------------------------------------------

ig_values = (
    X_centered
    *
    class_specific_weights
)


print(
    "\nIG matrix:"
)

print(
    ig_values.shape
)


assert (
    ig_values.shape
    ==
    X_train.shape
)


# ------------------------------------------------------------
# Check numerical stability
# ------------------------------------------------------------

assert np.all(
    np.isfinite(ig_values)
), (
    "IG values contain NaN or infinite values."
)


print(
    "✓ IG values calculated successfully"
)


# ============================================================
# 17. GLOBAL IG IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("STEP 5 — GLOBAL IG GENE IMPORTANCE")
print("=" * 70)


# ------------------------------------------------------------
# Mean absolute IG across ALL training samples
#
# One value per gene.
#
# Shape:
#
#       (15100,)
# ------------------------------------------------------------

mean_abs_ig = np.mean(
    np.abs(
        ig_values
    ),
    axis=0
)


print(
    "Mean absolute IG shape:",
    mean_abs_ig.shape
)


assert (
    mean_abs_ig.shape[0]
    ==
    len(gene_columns)
)


assert np.all(
    np.isfinite(mean_abs_ig)
)


print(
    "✓ Global IG importance calculated"
)


# ============================================================
# 18. CREATE IG RANKING TABLE
# ============================================================

print("\n" + "=" * 70)
print("STEP 6 — GLOBAL IG GENE RANKING")
print("=" * 70)


ig_ranking = pd.DataFrame({

    "gene":
        gene_columns,

    "mean_abs_ig":
        mean_abs_ig

})


ig_ranking = (
    ig_ranking
    .sort_values(
        "mean_abs_ig",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


ig_ranking["rank"] = (
    np.arange(
        len(ig_ranking)
    )
    + 1
)


print(
    "\nTop 20 IG-ranked genes:"
)


print(
    ig_ranking[
        [
            "rank",
            "gene",
            "mean_abs_ig"
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)


# ============================================================
# 19. SAVE COMPLETE IG RANKING
# ============================================================

IG_RANKING_FILE = (
    OUTPUT_DIR /
    "ig_gene_ranking.csv"
)


ig_ranking.to_csv(
    IG_RANKING_FILE,
    index=False
)


print(
    "\nSaved complete IG ranking:"
)

print(
    IG_RANKING_FILE
)


# ============================================================
# 20. SAVE IG INFORMATION
# ============================================================

IG_INFO_FILE = (
    OUTPUT_DIR /
    "ig_info.json"
)


ig_info = {

    "method":
        "Logistic Regression + Integrated Gradients",

    "gene_selection_data":
        "ALL training samples",

    "number_of_training_samples":
        int(
            X_train.shape[0]
        ),

    "number_of_genes":
        int(
            X_train.shape[1]
        ),

    "number_of_classes":
        int(
            len(cancer_classes)
        ),

    "baseline":
        "mean expression of all training samples",

    "attribution_target":
        "true cancer class",

    "aggregation":
        "mean absolute IG across all training samples",

    "top_k_sequence_lengths":
        TOP_K,

    "geneformer_genes_per_sequence":
        N_GENES,

    "geneformer_max_length":
        MAX_LENGTH,

    "random_state":
        RANDOM_STATE

}


with open(
    IG_INFO_FILE,
    "w"
) as f:

    json.dump(
        ig_info,
        f,
        indent=4
    )


print(
    "Saved IG information:"
)

print(
    IG_INFO_FILE
)



STEP 3 — DEFINE INTEGRATED GRADIENTS BASELINE
IG baseline shape: (15100,)
Baseline mean: 6.935503959655762
Baseline minimum: 0.0
Baseline maximum: 16.43492317199707
✓ IG baseline verified
Saved IG baseline:
F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\ig_baseline_training_mean.npy

STEP 4 — CALCULATE INTEGRATED GRADIENTS
Using ALL training samples:
Number of samples: 2411
Number of genes: 15100
Number of classes: 5

Class-specific coefficient matrix:
(2411, 15100)

Centered training matrix:
(2411, 15100)

IG matrix:
(2411, 15100)
✓ IG values calculated successfully

STEP 5 — GLOBAL IG GENE IMPORTANCE
Mean absolute IG shape: (15100,)
✓ Global IG importance calculated

STEP 6 — GLOBAL IG GENE RANKING

Top 20 IG-ranked genes:
 rank            gene  mean_abs_ig
    1 ENSG00000168878     0.083019
    2 ENSG00000142515     0.080737
    3 ENSG00000110484     0.069714
    4 ENSG00000167751     0.064890
    5 ENSG00000129824     0.060802
    6 ENSG000001

In [22]:
# ============================================================
# 21. PRECOMPUTE GENEFORMER TOKEN IDS
# ============================================================

print("\n" + "=" * 70)
print("STEP 7 — PRECOMPUTE GENEFORMER TOKEN IDS")
print("=" * 70)


gene_token_ids = {

    gene:
        token_dictionary[gene]

    for gene in gene_columns

}


print(
    "Precomputed token IDs for",
    len(gene_token_ids),
    "genes."
)


assert (
    len(gene_token_ids)
    ==
    len(gene_columns)
)


print(
    "✓ Token IDs verified"
)


# ============================================================
# 22. CREATE IG GENE LISTS
# ============================================================

print("\n" + "=" * 70)
print("STEP 8 — CREATE TOP IG GENE LISTS")
print("=" * 70)


selected_gene_lists = {}


for k in N_GENES:

    selected_genes = (

        ig_ranking
        .head(k)
        [
            "gene"
        ]
        .tolist()

    )


    selected_gene_lists[k] = (
        selected_genes
    )


    print(
        f"\nTop {k} IG genes:"
    )


    print(
        selected_genes[:10],
        "..."
    )


    # --------------------------------------------------------
    # Save selected gene list
    # --------------------------------------------------------

    gene_file = (

        OUTPUT_DIR
        /
        f"top_{k}_ig_genes.csv"

    )


    pd.DataFrame({

        "rank":
            np.arange(
                1,
                k + 1
            ),

        "gene":
            selected_genes

    }).to_csv(

        gene_file,

        index=False

    )


    print(
        "Saved:",
        gene_file
    )


# ============================================================
# 23. TOKENIZATION FUNCTION
# ============================================================

def tokenize_dataframe(
    df,
    dataset_name,
    selected_genes,
    target_k
):

    print("\n" + "=" * 70)

    print(
        f"Tokenizing {dataset_name} "
        f"using TOP {target_k} IG GENES"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Select only IG-ranked genes
    # --------------------------------------------------------

    X = df[
        selected_genes
    ].to_numpy(
        dtype=np.float32
    )


    input_ids = []

    labels = []

    cancer_types = []

    tissue_types = []


    # --------------------------------------------------------
    # Process samples
    # --------------------------------------------------------

    for i in tqdm(
        range(
            X.shape[0]
        ),
        desc=dataset_name
    ):

        values = X[i]


        # ----------------------------------------------------
        # Check values
        # ----------------------------------------------------

        if not np.all(
            np.isfinite(values)
        ):

            raise ValueError(

                f"{dataset_name}, "
                f"sample {i}: "
                "contains NaN or infinite values."

            )


        # ----------------------------------------------------
        # Geneformer expression ranking
        #
        # IG determines WHICH genes are retained.
        #
        # Geneformer then ranks these genes according
        # to their expression for each individual sample.
        # ----------------------------------------------------

        order = np.argsort(

            -values,

            kind="stable"

        )


        ranked_genes = [

            selected_genes[j]

            for j in order

        ]


        # ----------------------------------------------------
        # Convert genes to Geneformer token IDs
        # ----------------------------------------------------

        ranked_tokens = np.array(

            [

                gene_token_ids[
                    gene
                ]

                for gene
                in ranked_genes

            ],

            dtype=np.int32

        )


        # ----------------------------------------------------
        # Construct sequence:
        #
        # CLS
        # +
        # ranked genes
        # +
        # EOS
        # ----------------------------------------------------

        sequence = np.concatenate(

            [

                np.array(
                    [CLS_TOKEN],
                    dtype=np.int32
                ),

                ranked_tokens,

                np.array(
                    [EOS_TOKEN],
                    dtype=np.int32
                )

            ]

        )


        # ----------------------------------------------------
        # Safety checks
        # ----------------------------------------------------

        expected_length = (
            target_k + 2
        )


        assert (
            len(sequence)
            ==
            expected_length
        )


        assert (
            sequence[0]
            ==
            CLS_TOKEN
        )


        assert (
            sequence[-1]
            ==
            EOS_TOKEN
        )


        # ----------------------------------------------------
        # Save sequence
        # ----------------------------------------------------

        input_ids.append(
            sequence.tolist()
        )


        # ----------------------------------------------------
        # Cancer label
        # ----------------------------------------------------

        cancer_type = (
            df.iloc[i][
                "Cancer_Type"
            ]
        )


        labels.append(
            label_to_id[
                cancer_type
            ]
        )


        cancer_types.append(
            cancer_type
        )


        # ----------------------------------------------------
        # Tissue type
        # ----------------------------------------------------

        tissue_types.append(
            df.iloc[i][
                "tissue_type"
            ]
        )


    # ========================================================
    # CREATE HUGGING FACE DATASET
    # ========================================================

    dataset_dict = {

        "input_ids":
            input_ids,

        "labels":
            labels,

        "Cancer_Type":
            cancer_types,

        "tissue_type":
            tissue_types

    }


    dataset = Dataset.from_dict(
        dataset_dict
    )


    return dataset


# ============================================================
# 24. CREATE FOUR GENEFORMER DATASETS
# ============================================================

print("\n" + "=" * 70)
print("STEP 9 — CREATE GENEFORMER DATASETS")
print("=" * 70)


for k in N_GENES:

    print("\n\n")

    print(
        "#" * 70
    )

    print(
        f"PROCESSING IG TOP {k}"
    )

    print(
        "#" * 70
    )


    selected_genes = (
        selected_gene_lists[k]
    )


    # ========================================================
    # TRAIN
    # ========================================================

    train_dataset = tokenize_dataframe(

        train_df,

        "TRAIN",

        selected_genes,

        k

    )


    # ========================================================
    # VALIDATION
    # ========================================================

    validation_dataset = tokenize_dataframe(

        val_df,

        "VALIDATION",

        selected_genes,

        k

    )


    # ========================================================
    # TEST
    # ========================================================

    test_dataset = tokenize_dataframe(

        test_df,

        "TEST",

        selected_genes,

        k

    )


    # ========================================================
    # DATASET DICT
    # ========================================================

    tokenized_dataset = DatasetDict({

        "train":
            train_dataset,

        "validation":
            validation_dataset,

        "test":
            test_dataset

    })


    # ========================================================
    # OUTPUT DIRECTORY
    # ========================================================

    FINAL_OUTPUT = (

        OUTPUT_DIR
        /
        f"tcga_geneformer_ig_top{k}"

    )


    # ========================================================
    # SAVE
    # ========================================================

    tokenized_dataset.save_to_disk(

        str(
            FINAL_OUTPUT
        )

    )


    print(
        "\nSaved:"
    )

    print(
        FINAL_OUTPUT
    )


    # ========================================================
    # VERIFICATION
    # ========================================================

    print(
        "\nDataset:"
    )

    print(
        tokenized_dataset
    )


    print(
        "\nSizes:"
    )


    print(
        "Train:",
        len(
            tokenized_dataset[
                "train"
            ]
        )
    )


    print(
        "Validation:",
        len(
            tokenized_dataset[
                "validation"
            ]
        )
    )


    print(
        "Test:",
        len(
            tokenized_dataset[
                "test"
            ]
        )
    )


    # --------------------------------------------------------
    # First sample
    # --------------------------------------------------------

    sample = (
        tokenized_dataset[
            "train"
        ][0]
    )


    print(
        "\nFirst sample:"
    )


    print(
        "Cancer:",
        sample[
            "Cancer_Type"
        ]
    )


    print(
        "Label:",
        sample[
            "labels"
        ]
    )


    print(
        "Sequence length:",
        len(
            sample[
                "input_ids"
            ]
        )
    )


    print(
        "First tokens:",
        sample[
            "input_ids"
        ][:10]
    )


    print(
        "Last tokens:",
        sample[
            "input_ids"
        ][-10:]
    )


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print(
    "INTEGRATED GRADIENTS + GENEFORMER PIPELINE COMPLETE"
)
print("=" * 70)


print(
    "\nIG ranking:"
)

print(
    IG_RANKING_FILE
)


print(
    "\nGenerated Geneformer datasets:"
)


for k in N_GENES:

    print(

        f"  TOP {k} genes: "
        f"{OUTPUT_DIR / f'tcga_geneformer_ig_top{k}'}"

    )


print(
    "\n✓ Logistic Regression trained on TRAINING DATA only"
)

print(
    "✓ IG calculated using ALL training samples"
)

print(
    "✓ Validation data NOT used for gene selection"
)

print(
    "✓ Test data NOT used for gene selection"
)

print(
    "✓ Training-set mean used as IG baseline"
)

print(
    "✓ True cancer class used for attribution"
)

print(
    "✓ Mean absolute IG used for global gene ranking"
)

print(
    "✓ Top 510 genes selected"
)

print(
    "✓ Top 1022 genes selected"
)

print(
    "✓ Top 2046 genes selected"
)

print(
    "✓ Top 4094 genes selected"
)

print(
    "✓ Same IG gene lists applied to train/validation/test"
)

print(
    "✓ Geneformer expression ranking completed"
)

print(
    "✓ Hugging Face datasets saved"
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

print(
    "\n" + "=" * 70
)


STEP 7 — PRECOMPUTE GENEFORMER TOKEN IDS
Precomputed token IDs for 15100 genes.
✓ Token IDs verified

STEP 8 — CREATE TOP IG GENE LISTS

Top 510 IG genes:
['ENSG00000168878', 'ENSG00000142515', 'ENSG00000110484', 'ENSG00000167751', 'ENSG00000129824', 'ENSG00000185303', 'ENSG00000122852', 'ENSG00000105388', 'ENSG00000131400', 'ENSG00000101076'] ...
Saved: F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\top_510_ig_genes.csv

Top 1022 IG genes:
['ENSG00000168878', 'ENSG00000142515', 'ENSG00000110484', 'ENSG00000167751', 'ENSG00000129824', 'ENSG00000185303', 'ENSG00000122852', 'ENSG00000105388', 'ENSG00000131400', 'ENSG00000101076'] ...
Saved: F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\top_1022_ig_genes.csv

Top 2046 IG genes:
['ENSG00000168878', 'ENSG00000142515', 'ENSG00000110484', 'ENSG00000167751', 'ENSG00000129824', 'ENSG00000185303', 'ENSG00000122852', 'ENSG00000105388', 'ENSG00000131400', 'ENSG0000010107

TRAIN: 100%|██████████| 2411/2411 [00:25<00:00, 95.06it/s] 



Tokenizing VALIDATION using TOP 510 IG GENES


VALIDATION: 100%|██████████| 517/517 [00:04<00:00, 103.84it/s]



Tokenizing TEST using TOP 510 IG GENES


...


Tokenizing VALIDATION using TOP 4094 IG GENES


VALIDATION: 100%|██████████| 517/517 [00:06<00:00, 84.95it/s] 



Tokenizing TEST using TOP 4094 IG GENES


TEST: 100%|██████████| 517/517 [00:05<00:00, 91.43it/s] 



Saved:
F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\tcga_geneformer_ig_top4094

Dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 2411
    })
    validation: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 517
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 517
    })
})

Sizes:
Train: 2411
Validation: 517
Test: 517

First sample:
Cancer: PRAD
Label: 4
Sequence length: 4096
First tokens: [2, 8128, 11973, 14988, 10099, 10219, 2899, 1272, 3558, 11801]
Last tokens: [14945, 13113, 2415, 13506, 19757, 12701, 9475, 13519, 15504, 3]

INTEGRATED GRADIENTS + GENEFORMER PIPELINE COMPLETE

IG ranking:
F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\ig_gene_ranking.csv

Generated Geneformer datasets:
  TOP 510 genes: